# ImBrAid anonymization

Create the anonymized `random_id` tree, then create the deterministic `time_shifted` tree.

In [1]:
from pathlib import Path
import sys

START = Path.cwd().resolve()
PROJECT_ROOT = next(
    path for path in (START,) + tuple(START.parents)
    if (path / 'helpers').is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT))

from helpers.anonymization import (
    build_export_plan, build_generated_participant_plan,
    build_time_shift_plan, create_or_load_mapping,
    discover_et_recordings, discover_source_entries,
    execute_export, execute_time_shift, load_config, summarize_plan,
)

CONFIG_PATH = PROJECT_ROOT / 'ImBrAid' / 'config.json'
CONFIG = load_config(CONFIG_PATH)
EXPORT = False
RUN_TIME_SHIFT = False

In [2]:
entries = discover_source_entries(CONFIG)
recordings = discover_et_recordings(CONFIG)
records = ({entry.record for entry in entries} |
           {recording.record for recording in recordings})
mapping = create_or_load_mapping(CONFIG, records, commit=EXPORT)
plan = build_export_plan(entries, mapping, recordings)
generated_participants = build_generated_participant_plan(
    recordings, plan, mapping
)
summarize_plan(entries, plan, generated_participants, records)

{'records': 43,
 'files': 668,
 'bytes': 109408944087,
 'modalities': {'Driver Video': {'files': 43, 'bytes': 10052665327},
  'ET': {'files': 433, 'bytes': 80407219490},
  'IDUN': {'files': 106, 'bytes': 881439788},
  'LSL': {'files': 43, 'bytes': 10472760950},
  'Silab': {'files': 43, 'bytes': 7594858532}},
 'generated_participant_files': 41,
 'target_paths': 709}

In [3]:
if EXPORT:
    execute_export(CONFIG, plan, generated_participants)
    for entry in plan:
        if entry.target_relative.parts[1] == 'Silab':
            path = CONFIG.output_root / entry.target_relative
            path.write_bytes(path.read_bytes().replace(
                b'\nData,', b'\n' + entry.random_id.encode() + b','
            ))

In [4]:
if RUN_TIME_SHIFT:
    time_shift_plan = build_time_shift_plan(CONFIG.output_root)
    time_shift_summary = execute_time_shift(
        time_shift_plan, CONFIG.time_shift_output_root
    )
    time_shift_summary